In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="09-similar-listing",
    notebook_name="01_similar_listing_system_design.ipynb"
)

# Similar Listing System Design

## The Big Idea (For a 12-Year-Old)

Imagine you are on Airbnb looking at a cozy treehouse in the mountains. At the bottom of the page, you see a section that says **"More places like this."** It shows you other cool treehouses, cabins, and mountain homes that are somehow... *similar*.

But how does the computer know which houses are similar? Nobody sat down and manually compared 5 million homes. Instead, the computer watches what people DO. If thousands of people look at Treehouse A and then also look at Cabin B, the computer figures out: "Hey, these two places must be similar -- people keep looking at them together!"

It is like how a librarian notices that kids who pick up Harry Potter also tend to pick up Percy Jackson. She does not need to read both books to know they are similar -- she just watches the pattern.

---

## Staff-Level Technical Summary

We design a **session-based similar listing recommendation system** (Airbnb-style) using:
- **Listing2Vec**: Word2Vec-inspired embeddings trained on user click sessions
- **Skip-gram with negative sampling** adapted for listing co-occurrence
- **Improved loss function**: global context (booked listing) + hard negatives (same region)
- **Three-pipeline architecture**: training, indexing, prediction
- **Offline metric**: average rank of eventually booked listing | **Online metrics**: CTR, session book rate

## 1. Problem Definition and Clarifying Requirements

### The Interview Starts Here

In a real ML system design interview, you always start by asking clarifying questions. Never jump straight into the solution.

In [ ]:
# Organize the clarifying requirements as structured data

requirements = {
    "business_objective": "Increase the number of bookings",
    "similarity_definition": "Same neighborhood, city, price range, etc.",
    "personalization": "Treat logged-in and anonymous users equally (simplified)",
    "num_listings": "~5 million",
    "training_data": "User-listing interactions only (no listing/user attributes)",
    "cold_start": "New listings can appear in recommendations after 1 day",
    "system_io": {
        "input": "A listing the user is currently viewing (listing ID)",
        "output": "A ranked list of similar listings sorted by P(click)"
    }
}

print("=== Similar Listing System Requirements ===")
print(f"\nBusiness Objective: {requirements['business_objective']}")
print(f"Scale: {requirements['num_listings']} listings")
print(f"Similarity: {requirements['similarity_definition']}")
print(f"\nInput:  {requirements['system_io']['input']}")
print(f"Output: {requirements['system_io']['output']}")
print(f"\nKey Constraint: Model uses ONLY browsing behavior (co-occurrence),")
print(f"NOT listing attributes like price, location, or amenities.")
print(f"The embeddings IMPLICITLY learn these relationships from behavioral data.")

## 2. Why Session-Based Recommendation?

### The Key Insight

**Simple explanation**: Traditional recommendation systems are like a friend who has known you for years -- they recommend based on your long-term taste. Session-based systems are like a smart store assistant who watches what you are looking at RIGHT NOW and suggests similar things in the moment.

For vacation rentals, what you clicked on 5 minutes ago is much more useful than what you searched for 6 months ago. If you are currently browsing beach houses in Miami, your ski cabin search from last winter is irrelevant.

| Aspect | Traditional RecSys | Session-Based RecSys |
|--------|-------------------|---------------------|
| User interest model | Long-term, stable | Short-term, dynamic |
| Key signal | Historical interactions (months/years) | Current browsing session (minutes/hours) |
| Goal | Learn generic user interests | Understand current intent |
| Best for | Netflix, Spotify | Airbnb, Amazon "also viewed" |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Visualize: Session-based vs Traditional recommendation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Traditional: uses long-term history
ax = axes[0]
months = ['6 mo ago', '5 mo ago', '4 mo ago', '3 mo ago', '2 mo ago', '1 mo ago', 'Now']
categories = ['Ski Cabin', 'Beach House', 'City Apt', 'Beach House', 'Mountain', 'Beach House', 'Beach House']
colors = ['#3498db', '#e74c3c', '#2ecc71', '#e74c3c', '#9b59b6', '#e74c3c', '#e74c3c']
ax.barh(range(len(months)), [1]*7, color=colors, height=0.6)
for i, (m, c) in enumerate(zip(months, categories)):
    ax.text(0.5, i, f"{m}: {c}", ha='center', va='center', fontsize=10, fontweight='bold')
ax.set_xlim(0, 1)
ax.set_yticks([])
ax.set_xticks([])
ax.set_title('Traditional RecSys\n(Uses ALL history -- months of data)', fontsize=12, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)

# Session-based: uses current session only
ax = axes[1]
clicks = ['Click 1: Beach Condo', 'Click 2: Ocean View Apt', 'Click 3: Beachfront Studio',
          'Click 4: Coastal Villa', 'Click 5: ???']
colors2 = ['#e74c3c', '#e74c3c', '#e74c3c', '#e74c3c', '#f39c12']
ax.barh(range(len(clicks)), [1]*5, color=colors2, height=0.6)
for i, c in enumerate(clicks):
    ax.text(0.5, i, c, ha='center', va='center', fontsize=10, fontweight='bold')
ax.set_xlim(0, 1)
ax.set_yticks([])
ax.set_xticks([])
ax.set_title('Session-Based RecSys\n(Uses CURRENT session only -- minutes of data)', fontsize=12, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.annotate('Predict next!', xy=(0.85, 4), fontsize=11, color='#e67e22', fontweight='bold')

plt.tight_layout()
plt.suptitle('Why Session-Based is Better for Vacation Rentals', fontsize=14, fontweight='bold', y=1.02)
plt.show()

## 3. The Embedding Approach: Listing2Vec

### The Core Idea

**Simple explanation**: Think of every listing as a dot on a huge map. But instead of a geographic map, it is a "similarity map." Houses that people tend to browse together get placed close together. To find similar listings, we just look for the nearest dots.

**Technical explanation**: We train a model that maps each listing to a dense embedding vector in d-dimensional space (e.g., d=32). Listings that frequently co-occur in user browsing sessions will have nearby vectors. This is directly inspired by **Word2Vec** -- but instead of words in sentences, we have listings in browsing sessions.

```
Word2Vec:     word     = listing
              sentence = browsing session
              context  = nearby clicks in the session
```

In [ ]:
# Simulate the embedding space concept
np.random.seed(42)

# Create clusters of listings in 2D embedding space
# Each cluster represents a type of listing
clusters = {
    'Miami Beach Houses': {'center': (2, 5), 'n': 8, 'color': '#e74c3c'},
    'NYC City Apartments': {'center': (7, 2), 'n': 8, 'color': '#3498db'},
    'Mountain Cabins': {'center': (5, 8), 'n': 6, 'color': '#2ecc71'},
    'LA Luxury Villas': {'center': (9, 7), 'n': 5, 'color': '#9b59b6'},
}

fig, ax = plt.subplots(1, 1, figsize=(10, 8))

for name, info in clusters.items():
    cx, cy = info['center']
    points_x = np.random.normal(cx, 0.5, info['n'])
    points_y = np.random.normal(cy, 0.5, info['n'])
    ax.scatter(points_x, points_y, c=info['color'], s=120, alpha=0.7, edgecolors='black', linewidth=0.5)
    ax.annotate(name, xy=(cx, cy), fontsize=11, fontweight='bold',
                ha='center', va='bottom',
                bbox=dict(boxstyle='round,pad=0.3', facecolor=info['color'], alpha=0.3))

# Show a query listing and its neighbors
query_x, query_y = 2.3, 5.2
ax.scatter([query_x], [query_y], c='gold', s=300, marker='*', edgecolors='black', linewidth=2, zorder=5)
ax.annotate('You are viewing\nTHIS listing', xy=(query_x, query_y), xytext=(query_x-1.5, query_y+1.5),
            fontsize=11, fontweight='bold', color='#d35400',
            arrowprops=dict(arrowstyle='->', color='#d35400', lw=2))

# Draw circle around nearest neighbors
circle = plt.Circle((query_x, query_y), 1.2, fill=False, linestyle='--', color='#d35400', linewidth=2)
ax.add_patch(circle)
ax.annotate('Similar listings\n(nearest neighbors)', xy=(query_x+1.0, query_y-0.8),
            fontsize=10, color='#d35400', style='italic')

ax.set_xlabel('Embedding Dimension 1', fontsize=12)
ax.set_ylabel('Embedding Dimension 2', fontsize=12)
ax.set_title('Listing Embedding Space\nSimilar listings cluster together', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Key insight: We never told the model about listing attributes (price, location, type).")
print("It learned these clusters purely from CO-OCCURRENCE in browsing sessions!")

## 4. Data Pipeline: From Raw Interactions to Search Sessions

### Available Data

We have three tables:
1. **Users** (user_id, name, age, etc.)
2. **Listings** (listing_id, host_id, price, city, type, etc.)
3. **User-Listing Interactions** (user_id, listing_id, interaction_type, timestamp)

The critical data is the **interactions table**. From it, we extract **search sessions**.

In [ ]:
import pandas as pd
from datetime import datetime, timedelta
import random

np.random.seed(42)
random.seed(42)

# === Simulate Raw Data ===

# Listings table
cities = ['Miami', 'Miami', 'Miami', 'NYC', 'NYC', 'NYC', 'LA', 'LA', 'Aspen', 'Aspen']
listing_types = ['Beach House', 'Beach Condo', 'Ocean Studio', 'City Apt', 'Loft', 'Penthouse',
                 'Villa', 'Bungalow', 'Ski Cabin', 'Mountain Lodge']
prices = [250, 180, 120, 300, 220, 500, 450, 200, 350, 280]

listings_df = pd.DataFrame({
    'listing_id': [f'L{i}' for i in range(1, 11)],
    'city': cities,
    'type': listing_types,
    'price': prices,
    'neighborhood': ['South Beach', 'South Beach', 'Miami Beach', 'Manhattan', 'Brooklyn', 'Manhattan',
                     'Beverly Hills', 'Santa Monica', 'Aspen Center', 'Aspen Highlands']
})

print("=== Listings ===")
print(listings_df.to_string(index=False))

# Simulate user search sessions
# A session = sequence of clicked listings + eventually booked listing
sessions = [
    {'session_id': 1, 'clicked': ['L1', 'L2', 'L3', 'L2', 'L1'], 'booked': 'L3'},  # Miami beach searcher
    {'session_id': 2, 'clicked': ['L4', 'L5', 'L6', 'L4'], 'booked': 'L5'},         # NYC searcher
    {'session_id': 3, 'clicked': ['L1', 'L3', 'L2'], 'booked': 'L1'},               # Miami beach searcher
    {'session_id': 4, 'clicked': ['L7', 'L8', 'L7'], 'booked': 'L8'},               # LA searcher
    {'session_id': 5, 'clicked': ['L9', 'L10', 'L9'], 'booked': 'L10'},             # Aspen searcher
    {'session_id': 6, 'clicked': ['L2', 'L1', 'L3', 'L1', 'L2'], 'booked': 'L2'},  # Miami beach searcher
    {'session_id': 7, 'clicked': ['L5', 'L4', 'L6', 'L5'], 'booked': 'L6'},         # NYC searcher
    {'session_id': 8, 'clicked': ['L4', 'L6', 'L5'], 'booked': 'L4'},               # NYC searcher
]

print("\n=== Search Sessions ===")
print(f"{'Session':<10} {'Clicked Listings':<35} {'Booked'}")
print("-" * 60)
for s in sessions:
    print(f"{s['session_id']:<10} {' -> '.join(s['clicked']):<35} {s['booked']}")

print(f"\nTotal sessions: {len(sessions)}")
print(f"\nNotice: Miami listings (L1, L2, L3) appear together in sessions,")
print(f"NYC listings (L4, L5, L6) appear together, etc.")
print(f"The model will learn that co-occurring listings are SIMILAR.")

## 5. Training Data Construction: Sliding Window + Negative Sampling

### How We Create Training Pairs

**Simple explanation**: We slide a window across each session. The listing in the middle of the window is the "center." Listings inside the window are "positive" (similar). We also randomly pick listings from outside -- those are "negative" (not similar).

**Technical explanation**: For each session, apply a sliding window of size `m`. For each window position:
- **Positive pairs** (label=1): (center listing, each context listing within window)
- **Negative pairs** (label=0): (center listing, randomly sampled listings outside window)

In [ ]:
def generate_training_pairs(sessions, all_listing_ids, window_size=2, num_negatives=2):
    """
    Generate positive and negative training pairs from search sessions.
    
    Think of it like this (for a 12-year-old):
    - You have a magnifying glass (window) that you slide over the session
    - The listing in the center of the magnifying glass is your 'focus'
    - Other listings you can see through the magnifying glass are 'similar'
    - Listings you CANNOT see are 'not similar'
    """
    positive_pairs = []
    negative_pairs = []
    
    for session in sessions:
        clicked = session['clicked']
        
        for center_idx in range(len(clicked)):
            center = clicked[center_idx]
            
            # Define context window boundaries
            start = max(0, center_idx - window_size)
            end = min(len(clicked), center_idx + window_size + 1)
            
            # Positive pairs: center with each context listing
            for ctx_idx in range(start, end):
                if ctx_idx != center_idx:
                    positive_pairs.append((center, clicked[ctx_idx], 1))
            
            # Negative pairs: center with random listings NOT in context
            context_set = set(clicked[start:end])
            available_negatives = [l for l in all_listing_ids if l not in context_set]
            
            for _ in range(num_negatives):
                if available_negatives:
                    neg = random.choice(available_negatives)
                    negative_pairs.append((center, neg, 0))
    
    return positive_pairs, negative_pairs


all_listing_ids = listings_df['listing_id'].tolist()
pos_pairs, neg_pairs = generate_training_pairs(sessions, all_listing_ids, window_size=2, num_negatives=2)

print("=== Training Pairs (first 10 positive, first 10 negative) ===")
print("\nPositive pairs (should be similar):")
for p in pos_pairs[:10]:
    print(f"  ({p[0]}, {p[1]}) -> label={p[2]}")

print("\nNegative pairs (should be dissimilar):")
for p in neg_pairs[:10]:
    print(f"  ({p[0]}, {p[1]}) -> label={p[2]}")

print(f"\nTotal positive pairs: {len(pos_pairs)}")
print(f"Total negative pairs: {len(neg_pairs)}")

In [ ]:
# Visualize the sliding window concept

fig, axes = plt.subplots(3, 1, figsize=(14, 7))

session_example = ['L1', 'L2', 'L3', 'L2', 'L1']
window_size = 2

for step, center_idx in enumerate([0, 2, 4]):
    ax = axes[step]
    
    for i, listing in enumerate(session_example):
        # Determine color
        if i == center_idx:
            color = '#e74c3c'  # Center = red
            label = 'Center'
        elif abs(i - center_idx) <= window_size:
            color = '#2ecc71'  # Context = green
            label = 'Context (+)'
        else:
            color = '#bdc3c7'  # Outside = grey
            label = 'Outside'
        
        rect = plt.Rectangle((i * 2, 0), 1.5, 0.8, facecolor=color, edgecolor='black', linewidth=1.5)
        ax.add_patch(rect)
        ax.text(i * 2 + 0.75, 0.4, listing, ha='center', va='center', fontsize=12, fontweight='bold')
    
    # Draw window bracket
    w_start = max(0, center_idx - window_size)
    w_end = min(len(session_example) - 1, center_idx + window_size)
    ax.plot([w_start * 2 - 0.1, w_end * 2 + 1.6], [0.9, 0.9], color='#2c3e50', linewidth=2)
    ax.plot([w_start * 2 - 0.1, w_start * 2 - 0.1], [0.85, 0.95], color='#2c3e50', linewidth=2)
    ax.plot([w_end * 2 + 1.6, w_end * 2 + 1.6], [0.85, 0.95], color='#2c3e50', linewidth=2)
    ax.text((w_start * 2 + w_end * 2 + 1.5) / 2, 1.05, f'Window (m={window_size})', ha='center', fontsize=10)
    
    ax.set_xlim(-0.5, 10)
    ax.set_ylim(-0.2, 1.3)
    ax.set_yticks([])
    ax.set_xticks([])
    ax.set_ylabel(f'Step {step+1}', fontsize=11, fontweight='bold')
    for spine in ax.spines.values():
        spine.set_visible(False)

# Legend
legend_elements = [
    mpatches.Patch(facecolor='#e74c3c', label='Center listing'),
    mpatches.Patch(facecolor='#2ecc71', label='Context (positive pairs)'),
    mpatches.Patch(facecolor='#bdc3c7', label='Outside window'),
]
axes[0].legend(handles=legend_elements, loc='upper right', fontsize=9)
axes[0].set_title('Sliding Window Over a Search Session: [L1, L2, L3, L2, L1]', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Model Architecture: Skip-Gram for Listings

### The Word2Vec Analogy

**Simple explanation**: Word2Vec learns that "king" and "queen" are similar because they appear in similar sentences. We do the same thing -- we learn that two beach houses are similar because they appear in the same browsing sessions.

**Technical explanation**: We use the **Skip-gram** architecture. Given a center listing, the model predicts which listings are in its context window. The model is a shallow neural network with:
- **Input**: one-hot encoded listing ID
- **Hidden layer**: embedding matrix (V x d), where V = vocab size, d = embedding dim
- **Output**: probability of each listing being in the context

### Loss Function

```
L = -SUM_{(c,p) in D_pos} log(sigmoid(E_c . E_p))
    -SUM_{(c,n) in D_neg} log(sigmoid(-E_c . E_n))
```

Where `E_c`, `E_p`, `E_n` are embedding vectors of center, positive, and negative listings.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class Listing2Vec(nn.Module):
    """
    Skip-gram model adapted for listing embeddings.
    
    12-year-old version:
    Each listing gets a secret code (a list of numbers). Two listings that
    people browse together get similar secret codes. The model learns these
    codes by looking at millions of browsing sessions.
    
    Staff-level detail:
    - Embedding layer maps listing IDs to dense vectors
    - Trained with negative sampling loss (not full softmax)
    - Dot product measures similarity between embeddings
    - Sigmoid converts dot product to probability
    """
    def __init__(self, num_listings, embedding_dim):
        super().__init__()
        # Each listing gets an embedding vector
        self.embeddings = nn.Embedding(num_listings, embedding_dim)
        # Initialize with small random values
        nn.init.xavier_uniform_(self.embeddings.weight)
    
    def forward(self, center_ids, context_ids):
        """
        Compute similarity between center and context listings.
        
        Returns: sigmoid(dot_product) = probability that context is a neighbor.
        """
        center_emb = self.embeddings(center_ids)    # (batch, embed_dim)
        context_emb = self.embeddings(context_ids)  # (batch, embed_dim)
        
        # Dot product -> similarity score
        dot_product = (center_emb * context_emb).sum(dim=1)
        
        # Sigmoid -> probability
        return torch.sigmoid(dot_product)
    
    def get_embedding(self, listing_id):
        """Get the learned embedding for a listing."""
        return self.embeddings(torch.tensor(listing_id)).detach().numpy()
    
    def get_all_embeddings(self):
        """Get all listing embeddings as a numpy array."""
        return self.embeddings.weight.detach().numpy()


# Create model
NUM_LISTINGS = len(all_listing_ids)
EMBEDDING_DIM = 16  # In production, typically 32-64

model = Listing2Vec(NUM_LISTINGS, EMBEDDING_DIM)

print(f"Model created:")
print(f"  Number of listings: {NUM_LISTINGS}")
print(f"  Embedding dimension: {EMBEDDING_DIM}")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters())}")
print(f"\nEmbedding matrix shape: {model.embeddings.weight.shape}")
print(f"  (Each row is one listing's embedding vector)")

In [ ]:
# === Train the Listing2Vec Model ===

# Build listing_id -> index mapping
listing_to_idx = {lid: idx for idx, lid in enumerate(all_listing_ids)}

# Prepare training data as tensors
all_pairs = pos_pairs + neg_pairs
random.shuffle(all_pairs)

centers = torch.tensor([listing_to_idx[p[0]] for p in all_pairs])
contexts = torch.tensor([listing_to_idx[p[1]] for p in all_pairs])
labels = torch.tensor([float(p[2]) for p in all_pairs])

# Training loop
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.BCELoss()

print("=== Training Listing2Vec ===")
print(f"Training samples: {len(all_pairs)} ({len(pos_pairs)} positive, {len(neg_pairs)} negative)")
print()

losses = []
for epoch in range(200):
    model.train()
    optimizer.zero_grad()
    
    predictions = model(centers, contexts)
    loss = criterion(predictions, labels)
    
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    if (epoch + 1) % 50 == 0:
        # Check accuracy
        with torch.no_grad():
            preds = (predictions > 0.5).float()
            accuracy = (preds == labels).float().mean()
        print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.4f} | Accuracy: {accuracy:.4f}")

# Plot training loss
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(losses, color='#3498db', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Listing2Vec Training Loss', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

# === Visualize the Learned Embedding Space ===

model.eval()
embeddings = model.get_all_embeddings()

# Reduce to 2D for visualization
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings)

# Color by city
city_colors = {'Miami': '#e74c3c', 'NYC': '#3498db', 'LA': '#2ecc71', 'Aspen': '#9b59b6'}

fig, ax = plt.subplots(figsize=(10, 8))

for i, listing_id in enumerate(all_listing_ids):
    city = listings_df[listings_df['listing_id'] == listing_id]['city'].values[0]
    listing_type = listings_df[listings_df['listing_id'] == listing_id]['type'].values[0]
    color = city_colors[city]
    
    ax.scatter(embeddings_2d[i, 0], embeddings_2d[i, 1], c=color, s=200, 
               edgecolors='black', linewidth=1.5, zorder=3)
    ax.annotate(f"{listing_id}\n({listing_type})", 
                xy=(embeddings_2d[i, 0], embeddings_2d[i, 1]),
                xytext=(5, 5), textcoords='offset points',
                fontsize=8, fontweight='bold')

# Legend
legend_elements = [mpatches.Patch(facecolor=c, label=city, edgecolor='black') 
                   for city, c in city_colors.items()]
ax.legend(handles=legend_elements, loc='upper right', fontsize=11)

ax.set_xlabel('PCA Dimension 1', fontsize=12)
ax.set_ylabel('PCA Dimension 2', fontsize=12)
ax.set_title('Learned Listing Embeddings (PCA Projection)\nListings from the same city cluster together!', 
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print cosine similarity matrix
sim_matrix = cosine_similarity(embeddings)
print("=== Cosine Similarity Matrix (top-3 similar for each listing) ===")
for i, lid in enumerate(all_listing_ids):
    sims = [(all_listing_ids[j], sim_matrix[i, j]) for j in range(NUM_LISTINGS) if i != j]
    sims.sort(key=lambda x: -x[1])
    top3 = sims[:3]
    city = listings_df[listings_df['listing_id'] == lid]['city'].values[0]
    print(f"  {lid} ({city:>5s}): {', '.join([f'{s[0]}({s[1]:.2f})' for s in top3])}")

## 7. Improving the Loss Function

The basic loss has **two shortcomings** that are critical interview talking points.

### Problem 1: No Signal from Booked Listings

**Simple explanation**: The basic training only teaches "these houses were clicked near each other." But clicking is not the same as BOOKING. We want to especially push each listing close to the one the user actually booked.

**Solution -- Global Context**: Treat the eventually booked listing as a global context that stays in every positive pair throughout the session. As the window slides, listings enter and leave the context, but the booked listing always stays.

### Problem 2: Easy Negatives from Different Regions

**Simple explanation**: If you randomly pick a "not similar" listing, it is probably in a totally different city. That is too easy! The hard question is: among all beach houses in Miami, which ones are similar?

**Solution -- Hard Negatives**: Sample negative listings from the SAME NEIGHBORHOOD as the center listing. These "hard negatives" force fine-grained distinctions.

### Updated Loss Function (4 Components)

```
L = -SUM(D_pos)  log(sig(E_c . E_p))       # Push center toward clicked neighbors
    -SUM(D_neg)  log(sig(-E_c . E_n))       # Push center away from random listings
    -SUM(D_book) log(sig(E_c . E_book))     # Push center toward booked listing (GLOBAL CONTEXT)
    -SUM(D_hard) log(sig(-E_c . E_hard))    # Push center away from same-region non-context (HARD NEGATIVES)
```

In [ ]:
# Visualize the two loss improvements

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Improvement 1: Global Context (Booked Listing) ---
ax = axes[0]
session_listings = ['L1', 'L2', 'L3', 'L2', 'L1']
booked = 'L3'

# Draw session timeline
for i, l in enumerate(session_listings):
    color = '#2ecc71' if l == booked else '#3498db'
    rect = plt.Rectangle((i * 1.5, 1.5), 1.2, 0.6, facecolor=color, edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(i * 1.5 + 0.6, 1.8, l, ha='center', va='center', fontsize=11, fontweight='bold')

# Draw booked listing below with arrows from each center
booked_rect = plt.Rectangle((3.0, 0), 1.2, 0.6, facecolor='#f1c40f', edgecolor='black', linewidth=2)
ax.add_patch(booked_rect)
ax.text(3.6, 0.3, f'{booked}\n(booked)', ha='center', va='center', fontsize=9, fontweight='bold')

# Arrows from each listing to booked
for i in range(len(session_listings)):
    ax.annotate('', xy=(3.6, 0.65), xytext=(i * 1.5 + 0.6, 1.45),
                arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=1.5, linestyle='--'))

ax.set_xlim(-0.5, 8)
ax.set_ylim(-0.5, 2.5)
ax.set_title('Improvement 1: Global Context\n(Booked listing always stays as positive)', fontsize=12, fontweight='bold')
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

# --- Improvement 2: Hard Negatives ---
ax = axes[1]

# Draw two regions
miami_circle = plt.Circle((2, 3), 1.8, fill=True, facecolor='#fadbd8', edgecolor='#e74c3c', linewidth=2, linestyle='--')
ax.add_patch(miami_circle)
ax.text(2, 4.5, 'Miami', fontsize=12, fontweight='bold', ha='center', color='#e74c3c')

nyc_circle = plt.Circle((6, 3), 1.8, fill=True, facecolor='#d4efdf', edgecolor='#27ae60', linewidth=2, linestyle='--')
ax.add_patch(nyc_circle)
ax.text(6, 4.5, 'NYC', fontsize=12, fontweight='bold', ha='center', color='#27ae60')

# Center listing
ax.scatter([2], [3], c='#e74c3c', s=200, zorder=5, edgecolors='black', linewidth=2)
ax.text(2, 2.5, 'Center\n(Miami)', ha='center', fontsize=9, fontweight='bold')

# Easy negative (NYC) -- too easy!
ax.scatter([6], [3], c='#27ae60', s=150, zorder=5, marker='x', linewidth=3)
ax.text(6, 2.5, 'Easy neg\n(NYC)', ha='center', fontsize=9, color='#27ae60')

# Hard negative (same region, different listing)
ax.scatter([2.8], [3.8], c='#e74c3c', s=150, zorder=5, marker='x', linewidth=3)
ax.text(2.8, 4.2, 'HARD neg\n(same region!)', ha='center', fontsize=9, color='#c0392b', fontweight='bold')

ax.set_xlim(-0.5, 8.5)
ax.set_ylim(0.5, 5.5)
ax.set_title('Improvement 2: Hard Negatives\n(Same region, NOT in context window)', fontsize=12, fontweight='bold')
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

## 8. Evaluation Metrics

### Offline Metric: Average Rank of Eventually Booked Listing

**Simple explanation**: After training, we test the model like this: for a real session where someone booked a listing, we ask the model to rank all listings by similarity to the first click. If the model puts the booked listing near the top, the model is good.

### Online Metrics

| Metric | Definition | Why It Matters |
|--------|-----------|----------------|
| **CTR** | % of recommended listings that get clicked | Measures engagement |
| **Session Book Rate** | % of sessions that end in a booking | Directly tied to business goal |

In [ ]:
def evaluate_average_rank(model, sessions, listing_to_idx, all_listing_ids):
    """
    Compute the average rank of the eventually booked listing.
    
    12-year-old version:
    For each session, we look at the first listing someone clicked.
    Then we ask the model: 'Rank all listings by similarity.'
    We check: how high did the BOOKED listing appear in that ranking?
    Lower rank number = better model.
    """
    model.eval()
    ranks = []
    
    with torch.no_grad():
        all_embeddings = model.get_all_embeddings()  # (num_listings, embed_dim)
        
        for session in sessions:
            first_click = session['clicked'][0]
            booked = session['booked']
            
            first_idx = listing_to_idx[first_click]
            booked_idx = listing_to_idx[booked]
            
            # Compute similarity between first click and all listings
            first_emb = all_embeddings[first_idx]
            similarities = cosine_similarity([first_emb], all_embeddings)[0]
            
            # Rank (descending similarity)
            ranked_indices = np.argsort(-similarities)
            
            # Find rank of booked listing (1-indexed)
            rank = np.where(ranked_indices == booked_idx)[0][0] + 1
            ranks.append(rank)
    
    return ranks


ranks = evaluate_average_rank(model, sessions, listing_to_idx, all_listing_ids)

print("=== Offline Evaluation: Average Rank of Booked Listing ===")
print(f"\n{'Session':<10} {'First Click':<15} {'Booked':<10} {'Rank of Booked':<15}")
print("-" * 55)
for i, session in enumerate(sessions):
    print(f"{session['session_id']:<10} {session['clicked'][0]:<15} {session['booked']:<10} {ranks[i]:<15}")

avg_rank = np.mean(ranks)
print(f"\nAverage Rank: {avg_rank:.2f} (out of {NUM_LISTINGS} listings)")
print(f"Lower is better. Perfect score = 1.0 (always ranks booked listing first).")

# Visualize
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, len(ranks)+1), ranks, color='#3498db', edgecolor='black', linewidth=0.5)
ax.axhline(y=avg_rank, color='#e74c3c', linestyle='--', linewidth=2, label=f'Average rank = {avg_rank:.1f}')
ax.set_xlabel('Session', fontsize=12)
ax.set_ylabel('Rank of Booked Listing', fontsize=12)
ax.set_title('How Well Does the Model Rank the Booked Listing?', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xticks(range(1, len(ranks)+1))
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 9. Serving Architecture

The production system has **three pipelines**:

```
TRAINING PIPELINE          INDEXING PIPELINE          PREDICTION PIPELINE
+------------------+      +------------------+      +----------------------+
| Daily retraining |----->| Pre-compute all  |----->| 1. Embedding Fetcher |
| on new sessions  |      | listing embeddings|     |    (lookup or cold   |
| and interactions |      | Store in index   |      |     start heuristic) |
+------------------+      +------------------+      | 2. Nearest Neighbor  |
                                                    |    (ANN search)      |
                                                    | 3. Re-Ranking        |
                                                    |    (filters, rules)  |
                                                    +----------------------+
```

### Key Components

1. **Embedding Fetcher**: Looks up pre-computed embedding. For new listings (cold start), uses embedding of a geographically nearby listing.

2. **Nearest Neighbor Service**: Uses Approximate Nearest Neighbor (ANN) search (FAISS, ScaNN, HNSW) to find similar listings in <10ms across 5M vectors.

3. **Re-Ranking Service**: Applies user filters (price, dates), removes unavailable listings, ensures diversity.

In [ ]:
# === Simulate the Full Prediction Pipeline ===

class SimilarListingPipeline:
    """
    End-to-end similar listing recommendation pipeline.
    
    12-year-old version:
    1. Look up the secret code (embedding) of the listing you are viewing
    2. Find other listings with the most similar secret codes
    3. Remove ones that do not match your filters (too expensive, wrong city)
    4. Show the top results!
    """
    def __init__(self, model, listings_df, listing_to_idx, all_listing_ids):
        self.model = model
        self.listings_df = listings_df
        self.listing_to_idx = listing_to_idx
        self.all_listing_ids = all_listing_ids
        
        # Pre-compute all embeddings (indexing pipeline)
        model.eval()
        self.all_embeddings = model.get_all_embeddings()
    
    def get_embedding(self, listing_id):
        """Step 1: Embedding Fetcher Service"""
        if listing_id in self.listing_to_idx:
            # Listing seen during training -> use learned embedding
            idx = self.listing_to_idx[listing_id]
            return self.all_embeddings[idx]
        else:
            # Cold start: use embedding of nearest known listing
            # In production: use geographically nearby listing
            print(f"  [Cold Start] {listing_id} not seen -- using nearest known listing")
            return self.all_embeddings[0]  # Simplified fallback
    
    def find_nearest(self, query_embedding, top_k=5):
        """Step 2: Nearest Neighbor Service (simplified brute force; use ANN in production)"""
        sims = cosine_similarity([query_embedding], self.all_embeddings)[0]
        ranked_indices = np.argsort(-sims)
        return [(self.all_listing_ids[i], sims[i]) for i in ranked_indices[:top_k]]
    
    def rerank(self, candidates, current_listing_id, max_price=None, same_city_only=True):
        """Step 3: Re-Ranking Service"""
        current_city = self.listings_df[self.listings_df['listing_id'] == current_listing_id]['city'].values[0]
        
        reranked = []
        for lid, sim in candidates:
            if lid == current_listing_id:
                continue  # Skip the listing itself
            
            row = self.listings_df[self.listings_df['listing_id'] == lid].iloc[0]
            
            if same_city_only and row['city'] != current_city:
                continue
            if max_price and row['price'] > max_price:
                continue
            
            reranked.append({
                'listing_id': lid,
                'similarity': sim,
                'city': row['city'],
                'type': row['type'],
                'price': row['price']
            })
        
        return reranked
    
    def recommend(self, listing_id, top_k=5, max_price=None, same_city_only=True):
        """Full pipeline: fetch -> search -> rerank"""
        emb = self.get_embedding(listing_id)
        candidates = self.find_nearest(emb, top_k=top_k + 5)  # Get extras for filtering
        results = self.rerank(candidates, listing_id, max_price, same_city_only)
        return results[:top_k]


# Build pipeline
pipeline = SimilarListingPipeline(model, listings_df, listing_to_idx, all_listing_ids)

# Test: Find listings similar to L1 (Miami Beach House)
print("=== Similar Listings for L1 (Miami Beach House, $250) ===")
print("With same_city_only=True:\n")
results = pipeline.recommend('L1', top_k=5, same_city_only=True)
for i, r in enumerate(results):
    print(f"  #{i+1}: {r['listing_id']} - {r['type']} ({r['city']}) ${r['price']}/night | sim={r['similarity']:.3f}")

print("\nWith same_city_only=False:\n")
results = pipeline.recommend('L1', top_k=5, same_city_only=False)
for i, r in enumerate(results):
    print(f"  #{i+1}: {r['listing_id']} - {r['type']} ({r['city']}) ${r['price']}/night | sim={r['similarity']:.3f}")

In [ ]:
# === Visualize recommendations for multiple query listings ===

query_listings = ['L1', 'L4', 'L9']
query_labels = ['Miami Beach House', 'NYC City Apt', 'Aspen Ski Cabin']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, qlid, qlabel in zip(axes, query_listings, query_labels):
    results = pipeline.recommend(qlid, top_k=4, same_city_only=False)
    
    names = [r['listing_id'] + '\n' + r['type'][:15] for r in results]
    sims = [r['similarity'] for r in results]
    colors = [city_colors.get(r['city'], '#95a5a6') for r in results]
    
    bars = ax.barh(range(len(names)), sims, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=9)
    ax.set_xlabel('Cosine Similarity', fontsize=10)
    ax.set_title(f'Similar to {qlid}\n({qlabel})', fontsize=11, fontweight='bold')
    ax.set_xlim(0, 1.1)
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3, axis='x')

plt.suptitle('Similar Listing Recommendations', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Additional Talking Points

These are bonus topics to bring up if time remains in the interview:

1. **Positional Bias**: Listings shown at the top of results get more clicks regardless of true relevance. Address with position-aware training or inverse propensity weighting.

2. **Random Walk Approaches**: Compare session-based embeddings to graph-based approaches (DeepWalk, Node2Vec). Random Walks with Restart (RWR) can capture both local and global graph structure.

3. **In-Session Personalization**: For logged-in users, incorporate longer-term booking history. Airbnb uses this to boost listings similar to past bookings.

4. **Seasonality**: Vacation rentals are highly seasonal (beach in summer, ski in winter). Incorporate temporal features or train separate seasonal models.

## Key Takeaways

1. **Session-based > Traditional** for vacation rentals -- current browsing session captures short-term intent better than long-term history
2. **Listing2Vec = Word2Vec for listings** -- listings are words, sessions are sentences, co-occurrence defines similarity
3. **Skip-gram + negative sampling** -- efficient training that scales to millions of listings
4. **Two critical loss improvements** -- global context (booked listing) and hard negatives (same region)
5. **Offline: average rank of booked listing** -- directly measures embedding quality for the booking task
6. **Online: CTR + session book rate** -- engagement plus business metric
7. **Three-pipeline serving** -- training (daily), indexing (pre-compute), prediction (fetch -> ANN search -> rerank)
8. **Cold start** -- use geographically nearby listing's embedding until enough data accumulates (1 day)

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)